# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the [FAIR^2 dataset](https://doi.org/10.71728/senscience.y7m0-f273) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library following a FAIR, schema-based approach.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset metadata with mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # The metadata is a structured object, not a dict/list

print(f"Dataset name: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Review and inspect available record sets and fields, referencing their `@id` fields as per the Croissant specification.

In [ ]:
# List all record sets available in the Croissant schema

record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record set(s) in the dataset.\n")
for i, record_set in enumerate(record_sets, 1):
    print(f"{i}. Record Set: {record_set['@id']}")
    print(f"   Name: {record_set.get('name')}")
    print(f"   Description: {record_set.get('description')}")
    print(f"   Fields:")
    for field in record_set.get('field', []):
        if isinstance(field, dict):
            field_id = field.get('@id')
            field_name = field.get('name')
        else:
            field_id = field
            field_name = None
        print(f"      - {field_id} {(f'({field_name})' if field_name else '')}")
    print()

### Record Set Sample
View a few sample records from a chosen record set using its `@id`.

In [ ]:
# Example: Print the first records from each available record set
for record_set in record_sets:
    record_set_id = record_set['@id']
    print(f"\nRecord Set: {record_set_id}")
    try:
        sample_records = list(dataset.records(record_set=record_set_id))
        if sample_records:
            for i, rec in enumerate(sample_records[:3]):
                print(f"  Record {i + 1}: {rec}")
        else:
            print("  No records found.")
    except Exception as e:
        print(f"  Error loading records: {e}")

## 3. Data Extraction
Load all data from record sets into pandas DataFrames for analysis. All references use each record set's `@id`, and then respective records are loaded.

In [ ]:
# Prepare DataFrames for each record set
dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets]

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame for record set '{record_set_id}' with shape {df.shape}.")
        else:
            print(f"No records found for record set '{record_set_id}'.")
    except Exception as e:
        print(f"Error loading DataFrame for '{record_set_id}': {e}")

# If there is at least one DataFrame loaded, inspect columns of the first one
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"\nColumns in '{first_rs_id}': {dataframes[first_rs_id].columns.tolist()}")
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
We will filter, normalize, and group the data using numerical and categorical fields, referencing columns by their `@id`s from the previous step.

In [ ]:
# Choose a record set and numeric field by their @id

# Example: Let's use the first record set if available
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Analyze record set: {record_set_id}")
    
    # Display all column @id's
    print("Columns in this DataFrame:", df.columns.tolist())
    
    # Try to select a numeric field by checking dtypes or column names containing 'log', 'coeff', 'std', 'value', etc.
    numeric_candidates = [col for col in df.select_dtypes(include=[float, int]).columns if not col.lower().startswith('Unnamed')]
    if not numeric_candidates:
        numeric_candidates = [col for col in df.columns if any(s in col.lower() for s in ['log', 'coeff', 'std', 'value'])]
    
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        print(f"Using numeric field for processing: {numeric_field_id}")
        
        # Filter: For demonstration, set a threshold (use a quantile for generality)
        try:
            threshold = df[numeric_field_id].dropna().quantile(0.75)
            filtered_df = df[df[numeric_field_id] > threshold].copy()
            print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
            display(filtered_df.head())
            
            # Normalize
            normalized_col = f"{numeric_field_id}_normalized"
            filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            print(f"Normalized {numeric_field_id} (z-score) for filtered records:")
            display(filtered_df[[numeric_field_id, normalized_col]].head())
            
            # Try grouping by a likely categorical field (e.g., ward, region, gender, intervention type)
            group_candidates = [c for c in df.columns if any(s in c.lower() for s in ['ward', 'gender', 'region', 'intervention', 'group', 'county', 'type', 'category'])]
            group_field = group_candidates[0] if group_candidates else None
            if group_field:
                print(f"Grouping by: {group_field}")
                grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
                display(grouped_df.head())
            else:
                print("No suitable group field found for grouping.")
        except Exception as e:
            print(f"EDA failed: {e}")
    else:
        print("No numeric field found for EDA.")
else:
    print("No dataframes loaded for EDA.")

## 5. Visualization
Visualize distributions or relationships between key fields. Plots use `@id` column references.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'filtered_df' in locals() and not filtered_df.empty:
    # Histogram of the selected numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id} in Filtered Records")
    plt.show()

    # If grouping was possible, show a bar plot
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10, 5))
        sns.barplot(x=group_field, y=numeric_field_id, data=filtered_df)
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=25, ha='right')
        plt.show()
else:
    print("No suitable filtered records for visualization.")

## 6. Conclusion

- We have loaded the regression outputs and metadata from the FAIR^2 dataset using the machine-actionable Croissant schema.
- Record sets and fields were accessed by `@id` as per Croissant best practices.
- Basic exploratory and visual analyses were applied to regression outputs, highlighting how both numerical and group/categorical analyses are possible.
- This workflow can be automated and extended as the dataset or schema evolves thanks to the flexibility of schema-driven data loading.

For more information and advanced data processing, consult [mlcroissant documentation](https://github.com/mlcommons/croissant) and the [FAIR^2 dataset registry](https://sen.science/doi/10.71728/senscience.y7m0-f273/).